In [1]:
import h5py
import torch
import numpy as np
import os
import json

import keyrank_rs

import torch.nn as nn

from tqdm import tqdm

In [2]:
from data import get_fullkey_testloader

In [3]:
device = torch.device('cuda')

In [4]:
TRACE_START = 400
TRACE_END = 1500
SEED = 777

test_loader = get_fullkey_testloader(TRACE_START, TRACE_END, SEED)

N_KEYS = len(test_loader)

In [5]:
for trace, plaintext, key in test_loader:
    print(trace.shape)
    print(plaintext.shape)
    print(key.shape)
    break

torch.Size([1, 500, 1100])
torch.Size([1, 500, 2, 16])
torch.Size([1, 16])


In [6]:
def metadata_best_epoch(model_name) -> int:
    with open(f"models/{model_name}/metadata.json") as f:
        metadata = json.load(f)
        val_scores = metadata["scores"][1]
        best_epoch = np.array(val_scores).argmin()
    return best_epoch.item()

def get_traces_mean_std(trace_start, trace_end):
    """Load the mean and std of the training trace set within the given interval"""
    with open(f"misc/standardization/trace{trace_start}_{trace_end}.json") as f:
        info = json.load(f)
        mean = info['training_traces_mean']
        std = info['training_traces_std']

    return mean, std

In [52]:
IMPL = "fixslice"
ARCH = "zhang"
PREDICTION_TARGET = "2sbox"
TARGET_BYTE_IDX = 4

model_name = f"{IMPL}-{PREDICTION_TARGET}-byte{TARGET_BYTE_IDX}-{ARCH}-{TRACE_START}_{TRACE_END}-s{SEED}"

epoch = metadata_best_epoch(model_name)

model_path = f"models/{model_name}/epoch{epoch}.pt"
print(model_path)

model = torch.load(model_path).to()

TWO_WAY = True

models/fixslice-2sbox-byte4-zhang-400_1500-s777/epoch13.pt


/tmp/ipykernel_39221/150865125.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load(model_path).to()


In [53]:
"""Compute traces needed for 99% accurate full key recovery using single model"""

print(model_name)

MAX_TRACES = 100
log_softmax = nn.LogSoftmax(dim=1)

# Sample_idx, n_traces
success_matrix = torch.zeros(N_KEYS,MAX_TRACES)



for key_idx, (traces, plaintexts, key) in tqdm(enumerate(test_loader)):

    traces = traces[:, :MAX_TRACES].to(device).squeeze()
    plaintexts_B1 = plaintexts.squeeze()[:, 0] # first plaintext block
    plaintexts_B2 = plaintexts.squeeze()[:, 1] # second plaintext block
    true_key = key.long().squeeze()


    sbox1_scores, sbox2_scores = model(traces)
    numpy_scores1, numpy_scores2 = sbox1_scores.detach().cpu().numpy(), sbox2_scores.detach().cpu().numpy()

    full_guesses = torch.zeros(MAX_TRACES,16)

    for subkey in range(16):
        plaintext_B1_bytes = plaintexts_B1[:, subkey]
        plaintext_B1_bytes = plaintext_B1_bytes.long().detach().cpu().numpy().squeeze()

        plaintext_B2_bytes = plaintexts_B2[:, subkey]
        plaintext_B2_bytes = plaintext_B2_bytes.long().detach().cpu().numpy().squeeze()

        numpy_keyscores1 = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_B1_bytes, numpy_scores1)
        numpy_keyscores2 = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_B2_bytes, numpy_scores2)

        if TWO_WAY:
            numpy_keyscores3 = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_B1_bytes, numpy_scores2)
            numpy_keyscores4 = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_B2_bytes, numpy_scores1)
            x = torch.Tensor(numpy_keyscores1 + numpy_keyscores2 + numpy_keyscores3 + numpy_keyscores4)
        else:
            x = torch.Tensor(numpy_keyscores1 + numpy_keyscores2)
        x = log_softmax(x)
        x = x.cumsum(dim=0)
        guesses = x.argmax(dim=1)

        full_guesses[:, subkey] = guesses

    
    successes = (full_guesses.long() == true_key).all(dim=1)
    success_matrix[key_idx, :] = successes


full_key_traces = (success_matrix.sum(dim=0) >= 495).nonzero()[0].item() + 1

fixslice-2sbox-byte4-zhang-400_1500-s777


500it [00:08, 61.78it/s]


In [54]:
"""Compute traces needed for 99% accuracy on individual subkeys using single model"""


MAX_TRACES = 100

log_softmax = nn.LogSoftmax(dim=1)

# subkey, sample_idx, n_traces
success_matrix = torch.zeros(16,500,MAX_TRACES)

for key_idx, (traces, plaintexts, key) in tqdm(enumerate(test_loader)):

    traces = traces[:, :MAX_TRACES].to(device).squeeze()
    plaintexts_B1 = plaintexts.squeeze()[:, 0] # first plaintext block
    plaintexts_B2 = plaintexts.squeeze()[:, 1] # second plaintext block
    true_key = key.long().squeeze()

    sbox1_scores, sbox2_scores = model(traces)
    numpy_scores1, numpy_scores2 = sbox1_scores.detach().cpu().numpy(), sbox2_scores.detach().cpu().numpy()

    for subkey in range(16):
        plaintext_B1_bytes = plaintexts_B1[:, subkey]
        plaintext_B1_bytes = plaintext_B1_bytes.long().detach().cpu().numpy().squeeze()

        plaintext_B2_bytes = plaintexts_B2[:, subkey]
        plaintext_B2_bytes = plaintext_B2_bytes.long().detach().cpu().numpy().squeeze()

        numpy_keyscores1 = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_B1_bytes, numpy_scores1)
        numpy_keyscores2 = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_B2_bytes, numpy_scores2)

        if TWO_WAY:
            numpy_keyscores3 = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_B1_bytes, numpy_scores2)
            numpy_keyscores4 = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_B2_bytes, numpy_scores1)
            x = torch.Tensor(numpy_keyscores1 + numpy_keyscores2 + numpy_keyscores3 + numpy_keyscores4)
        else:
            x = torch.Tensor(numpy_keyscores1 + numpy_keyscores2)
        x = log_softmax(x)
        x = x.cumsum(dim=0)
        guesses = x.argmax(dim=1)

        success = (guesses.long() == true_key[subkey])
        success_matrix[subkey, key_idx] = success


n_traces_needed = []

for subkey in range(16):
    n_traces_needed.append((success_matrix[subkey].sum(dim=0) >= 495).nonzero()[0].item() + 1)

n_traces_needed

500it [00:08, 62.02it/s]


[29, 20, 23, 19, 24, 24, 19, 23, 23, 25, 19, 17, 43, 34, 33, 31]

In [55]:
TWO_WAY

True

In [56]:
#print("Subkey: &",  " & ".join([f"${n}$" for n in range(16)]), r"\\")
#print("\\hline")
#print("Mean traces: &", " & ".join([f"${mean:.01f}$" for mean in mean_traces_needed]), r"\\")
print("\\hline")
#print("\\nextrow")
print(f"{TARGET_BYTE_IDX} &", " & ".join([f"${n_traces:.0f}$" for n_traces in n_traces_needed]), f"& ${full_key_traces}$" r"\\")


#for idx, (mean, n99acc) in enumerate(zip(mean_traces_needed, traces_needed_99acc)):
#    print(f"Subkey {idx:02}, mean: {mean:.03f}, traces needed for 99%: {n99acc}")

\hline
4 & $29$ & $20$ & $23$ & $19$ & $24$ & $24$ & $19$ & $23$ & $23$ & $25$ & $19$ & $17$ & $43$ & $34$ & $33$ & $31$ & $43$\\
